# Lightweight CNN Baseline for Retinal OCT Disease Classification
© 2026 Utsab Saha, Puja Saha, MD Jahin Alam, and Maruf Ahmed.

This notebook contains the baseline PyTorch implementation used for retinal disease classification from Optical Coherence Tomography (OCT) images.

The aim of this notebook is to make the implementation easy to read and reproduce.

**Related paper**  
Utsab Saha, Puja Saha, MD Jahin Alam, and Maruf Ahmed, *Toward Efficient Identification of Retinal Diseases: A Lightweight Convolutional Neural Network-Based Approach Using Optical Coherence Tomography*, Healthcare Technology Letters, 2026. DOI: `https://doi.org/10.1049/htl2.70059`.

**Important note for readers**  
This notebook uses Kaggle-style dataset and weight paths. If you run it in a different environment, update the dataset path and saved-weight path before execution.


## Import required libraries

This cell loads the main PyTorch and torchvision modules used throughout the notebook. The implementation uses PyTorch for model definition/training, torchvision for image loading and preprocessing, and tqdm for progress bars during training and testing.


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

## Define the lightweight CNN architecture

This section defines the main model components:

- **SEBlock**: recalibrates channel-wise feature responses using squeeze-and-excitation attention.
- **LCBlock**: uses depthwise-style convolutional operations to reduce computational cost.
- **GLFBlock**: combines local convolutional processing with a global feature transformation.
- **CustomCNN**: stacks these blocks into the final retinal OCT classification network.



In [ ]:
import torch
import torch.nn as nn

# Define SE Block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size, channels, _, _ = x.size()
        squeeze = self.global_avg_pool(x).view(batch_size, channels)
        excitation = self.fc(squeeze).view(batch_size, channels, 1, 1)
        return x * excitation  # Channel-wise recalibration


# Define LC Block
class LCBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, activation=nn.Hardswish):
        super(LCBlock, self).__init__()
        self.use_residual = (in_channels == out_channels) and (stride == 1)
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * 6, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(in_channels * 6),
            activation(),
            nn.Conv2d(in_channels * 6, in_channels * 6, kernel_size=3, stride=stride, padding=1, groups=in_channels * 6, bias=False),
            nn.BatchNorm2d(in_channels * 6),
            activation(),
            nn.Conv2d(in_channels * 6, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        else:
            return self.block(x)


# Define GLF Block with SEBlock in the last layer
class GLFBlock(nn.Module):
    def __init__(self, in_channels, d, activation=nn.Hardswish):
        super(GLFBlock, self).__init__()
        self.local_block = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
            activation()
        )
        self.activation = activation()
        self.d = d
        self.se_block = SEBlock(in_channels)  # SE Block added

    def forward(self, x):
        local = self.local_block(x)
        batch_size, _, height, width = local.size()
        flattened_size = height * width * local.size(1)

        global_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, self.d),
            nn.ReLU(),
            nn.Linear(self.d, flattened_size),
            nn.Unflatten(1, (local.size(1), height, width))
        ).to(x.device)

        global_feat = global_block(local.view(batch_size, -1))
        global_feat = global_feat.view(batch_size, -1, height, width)

        # Apply SE Block before returning
        global_feat = self.se_block(global_feat)

        return x + global_feat


# Define Custom CNN
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.Hardswish(),
            LCBlock(16, 16, stride=1),
            LCBlock(16, 24, stride=2),
            LCBlock(24, 24, stride=1),
            LCBlock(24, 48, stride=2),
            GLFBlock(48, d=64),  # SE inside GLFBlock
            LCBlock(48, 64, stride=2),
            GLFBlock(64, d=80),  # SE inside GLFBlock
            LCBlock(64, 80, stride=2),
            GLFBlock(80, d=96),  # SE inside GLFBlock
            nn.Conv2d(80, 320, kernel_size=1, stride=1),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(320, 4)  # Adjusted classifier (number of classes)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            if isinstance(layer, nn.Conv2d):
                feature_maps = x  # Save the feature maps from the last Conv layer
        x = x.view(x.size(0), -1)  # Flatten
        return self.classifier(x)

## Check the model output shape

Before training, this quick sanity check passes a dummy OCT-like image batch through the model. The expected output shape is `[batch_size, number_of_classes]`, which confirms that the model produces one prediction vector per input image.


In [ ]:
# Test the model
model = CustomCNN()
input_tensor = torch.randn(32, 1, 224, 224)  # Batch size of 32, 1 channel, 224x224
output = model(input_tensor)
output.shape

## Count model parameters

This cell reports the total number of trainable and non-trainable parameters in the model. Since the goal of the work is lightweight retinal OCT classification, the parameter count is useful for comparing this model with larger CNN or transformer-based baselines.


In [ ]:
model = CustomCNN()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")

## Define image preprocessing

Each OCT image is converted to grayscale, resized to `224 × 224`, converted to a tensor, and normalized. This keeps the input format consistent with the model architecture and the training setup.


In [ ]:
# Data Preprocessing and Data Loaders
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

## Load training, validation, and test data

The notebook expects the dataset to be arranged in the standard `ImageFolder` format, where each class has its own subfolder inside `train`, `val`, and `test` directories.

Expected structure:

```text
OCT2017/
├── train/
│   ├── CNV/
│   ├── DME/
│   ├── DRUSEN/
│   └── NORMAL/
├── val/
│   ├── CNV/
│   ├── DME/
│   ├── DRUSEN/
│   └── NORMAL/
└── test/
    ├── CNV/
    ├── DME/
    ├── DRUSEN/
    └── NORMAL/
```

If the dataset is stored elsewhere, update `data_dir` in the code cell below.


In [ ]:
data_dir = "/kaggle/input/datasets/mislamshawon/oct2017"  # Change the dataset directory if necessary 
train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "train"), transform=transform)
val_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "val"), transform=transform)
test_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "test"), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Set device, loss function, and optimizer

This cell moves the model to GPU when available, defines cross-entropy loss for multi-class classification, and uses Adam optimizer for training.


In [ ]:
import torch.optim.lr_scheduler as lr_scheduler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Train and validate the model

The training loop performs the standard deep learning workflow:

1. Train the model on the training set.
2. Evaluate it on the validation set after each epoch.
3. Save the best model weights based on validation accuracy.
4. Reduce the learning rate when validation performance stops improving.

The lists `train_accuracies` and `val_accuracies` store accuracy values for later visualization or reporting.


In [ ]:
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=3)


import matplotlib.pyplot as plt

# Initialize lists to store accuracy values
train_accuracies = []
val_accuracies = []

# Best validation accuracy tracker
best_val_accuracy = 0.0

# Training and Validation Loop
epochs = 100
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    
    # Training
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
    
    train_accuracy = 100. * train_correct / train_total
    train_accuracies.append(train_accuracy)  # Record training accuracy
    print(f"Train Loss: {train_loss / len(train_loader):.4f}, Train Accuracy: {train_accuracy:.2f}%")
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_accuracy = 100. * val_correct / val_total
    val_accuracies.append(val_accuracy)  # Record validation accuracy
    print(f"Validation Loss: {val_loss / len(val_loader):.4f}, Validation Accuracy: {val_accuracy:.2f}%")
    
    # Check if current model is the best
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), "best_model_weight.pth")
        print(f"Best model saved with Validation Accuracy: {best_val_accuracy:.2f}%")
    
    # Step the scheduler
    scheduler.step(val_accuracy)
    print(f"Learning rate after epoch {epoch+1}: {scheduler.get_last_lr()[0]:.6f}")

## Load saved model weights (Optional)

This cell loads a previously saved PyTorch `state_dict`. The current path is written for a Kaggle environment. For GitHub users, this file will not be available automatically; they need to either train the model first or place the pretrained weight file at the expected location and update the path accordingly. In such case, reader can comment out this cell. 

In [ ]:
saved_weights_path = '/kaggle/working/best_model_weight.pth'  # Replace with the actual path to your saved weights file

# Load the state_dict from the saved file
model.load_state_dict(torch.load(saved_weights_path, weights_only=True))

## Evaluate the model on the test set

After training, this cell evaluates the model on the held-out test set and reports accuracy, precision, recall, and F1-score. These metrics provide a clearer picture of model performance than accuracy alone, especially when class distributions are not perfectly balanced.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize lists to store labels and predictions
all_labels = []
all_preds = []

# Testing
model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

# Calculate accuracy
test_accuracy = accuracy_score(all_labels, all_preds) * 100
precision = precision_score(all_labels, all_preds, average='weighted') * 100
recall = recall_score(all_labels, all_preds, average='weighted') * 100
f1 = f1_score(all_labels, all_preds, average='weighted') * 100

print(f"Test Loss: {test_loss / len(test_loader):.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall: {recall:.2f}%")
print(f"F1 Score: {f1:.2f}%")